# Markov Chains

---
In class today we will be implementing a Markov chain to process sentences

---
## Learning Objectives

1. Students will be able to explain the Markov Chain process
1. Implement a Markov Chain


Markov Chains represent a series of events following the Markov Property: future states are memory-less in that they depend only on the current state. This can be expanded to the idea of variable order Markov models where there is a variable-length memory (eg. 1st order Markov Model). Markov models consist of fully observable states. 

> A common example of this is in predicting the weather: We can clearly see the current weather and would like to predict tomorrow's weather. This is also applicable to biology with one case being CpG islands. 

Our goal today will be to implement a Markov model built from words. For our example text, we will use the classic example of Dr. Seuss because of the repetitive nature of the text.

---
## Train Markov model

For our initial implementation of the Markov Model, we will use the simple example of Dr. Seuss: "One fish two fish red fish blue fish."



In [2]:
def build_markov_model(markov_model, new_txt, order=1):
    
    if isinstance(new_txt, str):                #detect input type for standalone use
        words = new_txt.lower().split(" ")
    else:
        words = new_txt

    start_token = ['*S*'] * order                   #insert star and end
    end_token = ['*E*']
    words = start_token + words + end_token

    for i in range(len(words) - order):                 #iterate though words
        if order > 1:
            current_state = tuple(words[i:i + order])       # get current state
        elif order == 1:
            current_state = str(words[i])

        next_word = words[i + order]                    # get next word

        if current_state not in markov_model:           #update key in markov model
            markov_model[current_state] = {}                

        if next_word not in markov_model[current_state]:    #update count in key in markov model
            markov_model[current_state][next_word] = 0

        markov_model[current_state][next_word] += 1         # count all frequencies

    return markov_model

In [3]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print (markov_model)

{'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


###  Nth order Markov chain
In the above model, each event or word is output from only the previous state with no memory of any prior states. While this is useful in some cases, typical biological applications of Markov chains require higher-order models to accurately capture what we know about a system. For instance, in attempting to identify coding regions of a genome, we know that open reading frames (ORFs) contain codon triplets, and so a third or sixth order Markov chain would better describe these regions. Here you will implement a generalized form of our previous Markov Chain to allow for Nth order chains.


In [4]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text, order=2)
markov_model

{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 1},
 ('fish', 'blue'): {'fish': 1},
 ('blue', 'fish'): {'*E*': 1}}

## Generate text from Markov Model

Markov models are "generative models". That is, the probability states in the model can be used to generate output following the conditional probabilities in the model.

We will now generate a sequence of text from the Markov model. For this section, I recommend using np.random.choice, which allows for you to provide a probability distribution for drawing the next edge in the chain.

In [5]:
import numpy as np

def get_next_word(current_word, markov_model, seed=42):
    
    if seed is not None:
        np.random.seed(seed)
    
    if current_word in markov_model:                                #check current_word if in the markov_model
        next_words_dict = markov_model[current_word]                    # get next_words (dic)
        
        candidates = list(next_words_dict.keys())                       # extract the key(next_word)
        counts = list(next_words_dict.values())                         # extract the values(frenqucy)
        total_count = sum(counts)                                       # caculate total_count
        
        probabilities = [count / total_count for count in counts]       # caculate probobilitys for each word # P(word) = count(word) / total_count
        chosen_word = np.random.choice(candidates, p=probabilities)     # choose next word randomly based on probobilitys
        
        return chosen_word
    else:                                                               # handle if word not in model
        return None


  
def generate_sequence(markov_model, order=1, seed=42, max_len=10000):
    """
    generate a sequence from the markov_model
    """

    if not markov_model:
        return ""

    if order > 1:
        current_state = tuple(["*S*"] * order)      # set current_state
    else:
        current_state = "*S*"

    sentence = []
    next_word = ""
    i = 0
    while next_word != "*E*" and i < max_len:                                       # generate next_word and combine
        next_word = get_next_word(current_state, markov_model, seed=seed)           # use the get_next_word we made
        sentence.append(next_word)                                                  # append next_word to sentence
        if order > 1:
            current_state = current_state[1:] + (next_word,)                            # Sliding Window to remove old variable and add the new | transform next word to tuple
        elif order == 1:
            current_state = next_word
        i += 1

    return " ".join(sentence)                                                       # return combined sentence



def generate_random_text(markov_model=None, mode="line", path="one_fish_two_fish", order=1, seed=None, length=1):                #default parameters

    '''
    Multipurpose function that generates text based on the input type selected.
    If model is built on sentences, output sentence.
    If model is built on lines, output a line.
    If model is built on sonnets, output a sonnet (not perfect).

    Specify order and seed as parameters.

    NOTE: Sonnet only works on shakespeare. It does not count lines and just divides on empty lines instead.
    '''
    
    def split_sentences(text_content):
        '''
        Helper function to separate out sentences based on punctuation. Includes end quotations in the current string.
        '''
        text_content = text_content.replace("\n", " ")              #remove newlines to allow sentences to span lines
        split_text = [[]]                                           #nested list structure for segmnting text
        text_content = text_content.split(" ")                      #split words on spaces
        for i in range(len(text_content)):                          #loop logic
            if len(text_content[i]) > 0:
                split_text[0].append(text_content[i])
                if text_content[i][0] in {"!", "?", ".", '"'} and len(text_content[i]) == 1:
                    if i < len(text_content) - 1:
                        if text_content[i + 1] != '"':
                            split_text.insert(0, [])
        return split_text
    
    def format_output(text):
        '''
        Helper function to make the output look pretty. This includes removing spaces before punctuation and the start and end characters.
        Capitalizes first words and words after punctuation.
        Returns joined string.
        '''
        text = text.split(" ")
        for i in range(len(text) - 1, -1, -1):              #iterate thru the output-list backwards to avoid issues with changing length
            if text[i] in {"*S*", "*E*"}:
                text.pop(i)
            elif len(text[i]) == 1:                         #detect punctuation/single-letter words
                if ord(text[i]) in range(33, 64):           #use ASCII code for punctuation detection
                    text[i - 1] += text[i]
                    if i < len(text) - 1:
                        if text[i] in {"!", ".", "?"}:      #detect sentence ends
                            text[i + 1] = text[i + 1][0].capitalize() + text[i + 1][1:]     #beautify
                    text.pop(i)
                elif text[i] == "i":                        #capitalize FPP
                    text[i] = text[i].capitalize()
        output = " ".join(text)                             #join the output list
        if len(output) > 0:
            output = output[0].capitalize() + output[1:]
        return output

    if markov_model == None:
        if mode not in {"line", "sentence", "sonnet"}:          #ERROR DETECTION
            raise ValueError("WRONG MODE")

        try:
            with open(f"data/{path}.txt", "r") as f:        # open file
                text_content = f.read()                     # 2. read all line
        except FileNotFoundError:
            print(f"error: can't find the file {path} exist?")


        for p in {",", ".", "!", "?", "\"", ";", "--", ":"}:              # define punctuation
            text_content = text_content.replace(p, " " + p)         #  "," transform to " , " for split in markov_modle
        text_content = text_content.lower()

        if mode == "sentence":                                              #select the input segmentation mode specified by the user
            text_content = split_sentences(text_content)                    #sentence splitter is most complicated
        elif mode == "line":
            text_content = text_content.split("\n")                         #split by line is easiest
        elif mode == "sonnet":
            text_content = text_content.replace("\n", " \n ").split("  ")   #split by sonnet uses the double space and removes newlines
        text_content = [entry for entry in text_content if len(entry) > 0]  #remove empty sentences


        markov_model = {}
        for entry in text_content:
            markov_model = build_markov_model(markov_model, entry, order=order)     #build model. initializing markov_model as a separate variable maintains persistance outside the loop

    output_list = []
    for _ in range(length):
        output = generate_sequence(markov_model, order, seed=seed)                      # spawn output
        while len(output) == 0:
            output = generate_sequence(markov_model, order, seed=seed)                     # spawn output
        formatted_output = format_output(output)                                    # format output
        output_list.append(formatted_output)

    if length == 1:
        return output_list[0]
    else:
        return output_list

---

## All the Fish
Up till now, you have only been working with a line or two of the Dr. Seuss' _One Fish, Two Fish_. Now, I want you to build a model using the whole book and try different orders of Markov models.

In [6]:
# Now just add some more training data to the markov model. You can find it under data/one_fish_two_fish.txt

print (generate_random_text(mode="line", length=5, order=1))

['And my little bed at our mike, two feet stick out back again.', 'Some are low.', 'They walked all.', 'But a nook is yell. I like this is another.', 'One, three.']


## All possible outputs:

In [9]:
from pprint import pprint
for i in range(3):
    print("\nITERATION:", i + 1, "\n")
    print("\none first-order sentence from seuss\n")
    pprint(generate_random_text(mode="sentence", order=1, length=1))
    print("\nthree third-order sentences from seuss\n")
    pprint(generate_random_text(mode="sentence", order=3, length=3))
    print("\nthree third-order sentences from shakespeare\n")
    pprint(generate_random_text(mode="sonnet", path="sonnets", order=3, length=3))
    print("\nthree fifth-order sentences from the odyssey\n")
    pprint(generate_random_text(mode="sentence", path="odyssey", order=3, length=3))



ITERATION: 1 


one first-order sentence from seuss

'Look at home, so.'

three third-order sentences from seuss

['By the light of the moon, by the light of a star; they walked all night from '
 'near to far, from here to there, funny things are everywhere.',
 'I said hello.',
 'This is not right.']

three third-order sentences from shakespeare

['\n'
 ' unthrifty loveliness, why dost thou spend \n'
 " upon thy side, against myself i'll fight, \n"
 " after a thousand victories once foil'd, \n"
 ' is but the seemly raiment of my heart; \n'
 ' my bonds in thee are seen \n'
 ' to truths translated, and for myself mine own worth do define, \n'
 " as I by yours, you've pass'd a hell of time; \n"
 ' the first my thought, whose love to you, as you to me, then, and wish I '
 "were renew'd; \n"
 ' whilst, like a jewel (hung in ghastly night, \n'
 ' but, ah! Thought kills me that I am old, \n',
 ' hence, thou suborned informer! A true soul \n',
 '\n'
 " and having climb'd the steep-up heavenly